# AM — v0.3.7 vs v0.4.0 Data Loss Audit

**Author:** Aidan Meyers · Melaram Lab · TAMU-CC  
**Database:** Neon Postgres · project `aged-salad-62359207`  
**Schemas compared:** `aq_v0_3_7_epa` (EPA + TCEQ blend, historical) vs `aq` (TCEQ-only, current)  
**Last updated:** 2026-06-02  

Companion to `AM_Data_Availability_Audit.ipynb`. That notebook audits v0.4.0 in isolation. **This notebook diffs v0.3.7 against v0.4.0** so we can prove the EPA → TCEQ cutover did not silently lose data — and where it did, that the loss is explained by an architectural decision, not a regression.

### Why the comparison is non-trivial

The v0.4.0 rewrite intentionally restructured the data. A naive `v040.count - v037.count` would flag every architectural decision as a "loss":

| Architectural move (v0.4.0) | Naive diff sees | Real verdict |
|---|---|---|
| EPA AQS API path retired | EPA rows missing | **Expected** — TCEQ has the same physical-monitor data |
| VOCs split from `pollutant_hourly` into `vocs_1hr` + `vocs_24hr` | VOCs gone from `pollutant_hourly` | **Expected** — sum the two new tables back |
| Site 480290060 PM10 moved to `pollutant_daily_24hr` | PM10 gone for that site | **Expected** — add the 24hr table |
| 4 TSP-only sites + Von Ormy dropped from registry | 5 sites missing | **Expected** — outside scope |

So the audit asks **"is the TCEQ portion of v0.3.7 fully preserved in v0.4.0 after accounting for the moves above?"** — that's the only meaningful question. The EPA portion is, by design, no longer required (TCEQ reports the same physical monitor data without the EPA aggregation layer).

### What this notebook produces

1. **Side-by-side availability matrix** per `(aqsid × pollutant_group × year)` showing v0.3.7 EPA + TCEQ hours vs v0.4.0 hours.
2. **Verdict per row** — Match / Improved / EPA-only-removed / Decreased-unexpected / Out-of-scope-site / VOC-split-OK / PM10-routed-OK.
3. **Roll-ups** — total hours by pollutant, by site, by year — old vs new.
4. **Audit summary verdict** — explicit pass/fail on "did we lose any TCEQ data".
5. **Visual diff** — bar chart per pollutant with old vs new totals, color-coded by verdict.
6. **Investigation queue** — only rows flagged as `Decreased-unexpected` (these need human review).

All outputs land in `notebooks/reports/v037_vs_v040/`.

## 1. Setup + connection

In [ ]:
!pip install -q "psycopg[binary]" sqlalchemy pandas plotly matplotlib seaborn nbconvert

In [ ]:
import os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

BRAND_NAVY     = '#213c4e'
BRAND_ORANGE   = '#c2410c'
BRAND_LIGHT_BG = '#F5F7F9'
BRAND_OK       = '#2e7d4f'
BRAND_WARN     = '#e0a528'
BRAND_BAD      = '#c2410c'
BRAND_NEUTRAL  = '#9aa6ad'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   BRAND_LIGHT_BG,
    'axes.edgecolor':   BRAND_NAVY,
    'axes.labelcolor':  BRAND_NAVY,
    'xtick.color':      BRAND_NAVY,
    'ytick.color':      BRAND_NAVY,
    'axes.titlecolor':  BRAND_NAVY,
    'font.family':      'DejaVu Sans',
})

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPORT_DIR = (Path('reports') if IN_COLAB else Path.cwd() / 'reports') / 'v037_vs_v040'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / 'figs').mkdir(exist_ok=True)
print(f'OK report dir: {REPORT_DIR}')

In [ ]:
URL = None
try:
    from google.colab import userdata
    URL = userdata.get('AQ_POSTGRES_URL')
except Exception:
    pass
URL = URL or os.environ.get('AQ_POSTGRES_URL')
assert URL, 'Set AQ_POSTGRES_URL via Colab Secrets (key icon) or env var.'
if URL.startswith('postgresql://') and '+psycopg' not in URL:
    URL = 'postgresql+psycopg://' + URL[len('postgresql://'):]

engine = create_engine(URL, pool_pre_ping=True)
with engine.connect() as conn:
    ver = conn.execute(text('SELECT version()')).scalar()
print('OK connected:', ver[:80])

AUDIT_START_YEAR = 2015
AUDIT_END_YEAR   = 2025
# Tolerance: <= 1% difference counts as a Match (rounding from concurrent runs is fine)
MATCH_TOLERANCE_PCT = 1.0
# Architectural drops (v0.4.0 decisions documented in pipeline/docs/v0_4_0_migration.md)
DROPPED_SITES = {
    '480290623': 'TSP-only (decision #8)',
    '480290625': 'TSP-only (decision #8)',
    '480290626': 'TSP-only (decision #8)',
    '480291609': 'Calaveras Lake Park — TSP-only (decision #8)',
    '480291097': 'Von Ormy — not in TCEQ pull (decision #18)',
}
print(f'Audit window: {AUDIT_START_YEAR} -> {AUDIT_END_YEAR}')
print(f'Match tolerance: {MATCH_TOLERANCE_PCT}%')
print(f'Sites architecturally dropped in v0.4.0: {len(DROPPED_SITES)}')

## 2. v0.3.7 availability matrix — EPA vs TCEQ broken out

The v0.3.7 `aq_v0_3_7_epa.pollutant_hourly` table carries the `data_source` column (dropped in v0.4.0). Split the per-`(aqsid × pollutant_group × year)` count into EPA-sourced and TCEQ-sourced columns. The TCEQ slice is the apples-to-apples baseline for the v0.4.0 comparison; the EPA slice is what v0.4.0 deliberately retired.

In [ ]:
sql_v037 = text(f"""
    SELECT aqsid::text                              AS aqsid,
           site_name,
           county_name,
           pollutant_group,
           year::int                                AS year,
           SUM(CASE WHEN data_source='EPA'  THEN 1 ELSE 0 END) AS v037_epa_hours,
           SUM(CASE WHEN data_source='TCEQ' THEN 1 ELSE 0 END) AS v037_tceq_hours,
           COUNT(*)                                 AS v037_total_hours,
           COUNT(DISTINCT date_local)               AS v037_days,
           MIN(date_local)                          AS v037_first,
           MAX(date_local)                          AS v037_last
    FROM aq_v0_3_7_epa.pollutant_hourly
    WHERE sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group, year
""")
v037 = pd.read_sql(sql_v037, engine)
print(f'v0.3.7 (aqsid x pollutant x year) groups: {len(v037):,}')
print(f'  unique sites:           {v037.aqsid.nunique()}')
print(f'  pollutant_groups:       {sorted(v037.pollutant_group.unique())}')
print(f'  total EPA hours:        {v037.v037_epa_hours.sum():>12,}')
print(f'  total TCEQ hours:       {v037.v037_tceq_hours.sum():>12,}')
print(f'  total (EPA + TCEQ):     {v037.v037_total_hours.sum():>12,}')
v037.head()

## 3. v0.4.0 availability matrix — re-aggregated to v0.3.7 logical groups

v0.4.0 spreads the same logical pollutants across four tables: `pollutant_hourly`, `pollutant_daily_24hr`, `vocs_1hr`, `vocs_24hr`. For the comparison we **sum them back into the v0.3.7 logical groups**:

| v0.3.7 `pollutant_group` | v0.4.0 source(s) |
|---|---|
| `Ozone`, `NOx_Family`, `CO`, `SO2`, `PM2.5` | `pollutant_hourly` only |
| `PM10` | `pollutant_hourly` ∪ `pollutant_daily_24hr` (site 0060 lives in the 24hr table now) |
| `VOCs` | `vocs_1hr` ∪ `vocs_24hr` |

24hr tables are scaled `n_rows × 24` to express coverage on the same hourly axis (a 24-hour sampled day = 24 hours of expected coverage).

In [ ]:
# Apples-to-apples row-count semantics: v0.3.7's COUNT(*) counts every
# (aqsid, datetime, parameter_code, poc) row. To diff cleanly we use COUNT(*)
# on the v0.4.0 side too — including for VOCs (one row per chemical per
# datetime) and for pollutant_daily_24hr (which already stores one row per
# day, same as v0.3.7 stored its 24hr-sampled PM10).

# 3a. criteria pollutants (pollutant_hourly) — direct match
ph = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, site_name, county_name, pollutant_group, year::int AS year,
           COUNT(*) AS hours
    FROM aq.pollutant_hourly
    WHERE sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group, year
"""), engine)

# 3b. pollutant_daily_24hr (site 0060 PM10 today). v0.3.7 stored the same
# 24hr-sampled data as one row per day in pollutant_hourly, so COUNT(*) here
# matches COUNT(*) over there — NO ×24 multiplier.
pd24 = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, site_name, county_name, pollutant_group, year::int AS year,
           COUNT(*) AS hours
    FROM aq.pollutant_daily_24hr
    WHERE year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, site_name, county_name, pollutant_group, year
"""), engine)

# 3c. vocs_1hr — flatten pollutant_group to 'VOCs' to match v0.3.7's group label.
# Use COUNT(*) so the row-multiplicity (one row per chemical per hour) lines
# up with v0.3.7's row-count.
v1 = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, MAX(site_name) AS site_name, MAX(county_name) AS county_name,
           'VOCs' AS pollutant_group, year::int AS year,
           COUNT(*) AS hours
    FROM aq.vocs_1hr
    WHERE sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, year
"""), engine)

# 3d. vocs_24hr — same flatten, same COUNT(*)
v24 = pd.read_sql(text(f"""
    SELECT aqsid::text AS aqsid, MAX(site_name) AS site_name, MAX(county_name) AS county_name,
           'VOCs' AS pollutant_group, year::int AS year,
           COUNT(*) AS hours
    FROM aq.vocs_24hr
    WHERE sample_measurement IS NOT NULL
      AND year BETWEEN {AUDIT_START_YEAR} AND {AUDIT_END_YEAR}
    GROUP BY aqsid, year
"""), engine)

print('v0.4.0 sources:')
print(f'  pollutant_hourly groups:     {len(ph):>4}   ({ph.hours.sum():>15,} rows)')
print(f'  pollutant_daily_24hr groups: {len(pd24):>4}   ({pd24.hours.sum():>15,} rows)')
print(f'  vocs_1hr groups:             {len(v1):>4}   ({v1.hours.sum():>15,} rows)')
print(f'  vocs_24hr groups:            {len(v24):>4}   ({v24.hours.sum():>15,} rows)')

In [ ]:
# Concatenate + collapse on the (aqsid, pollutant_group, year) key. PM10 in
# pollutant_hourly + PM10 in pollutant_daily_24hr fold into a single PM10 row,
# and VOCs_1hr + VOCs_24hr fold into a single VOCs row — matching v0.3.7's
# logical grouping.
v040_all = pd.concat([ph, pd24, v1, v24], ignore_index=True)
v040 = (v040_all
    .groupby(['aqsid','pollutant_group','year'], as_index=False)
    .agg(v040_hours=('hours','sum'),
         site_name =('site_name','first'),
         county_name=('county_name','first')))
# Also flag which v0.4.0 table(s) the row came from for transparency
src_map = (v040_all.assign(t=lambda d: d.assign().pipe(lambda x: x))
                    .groupby(['aqsid','pollutant_group','year'])
                    .size()
                    .rename('n_v040_sources').reset_index())
v040 = v040.merge(src_map, on=['aqsid','pollutant_group','year'], how='left')
print(f'v0.4.0 collapsed matrix: {len(v040):,} groups')
print(f'  unique sites:        {v040.aqsid.nunique()}')
print(f'  pollutant_groups:    {sorted(v040.pollutant_group.unique())}')
print(f'  total hours:         {v040.v040_hours.sum():,}')
v040.head()

## 4. Outer-join the two matrices and classify each row

Full outer join on `(aqsid, pollutant_group, year)`. Anything in v0.3.7 but not v0.4.0 → left side only. Anything new in v0.4.0 → right side only. Everything else → both sides present, compare directly.

**Verdict logic** (in priority order — first match wins):

1. **`Out-of-scope-site`** — `aqsid` is in `DROPPED_SITES` (TSP-only or Von Ormy). Expected gone.
2. **`EPA-only-removed`** — v0.3.7 had only EPA-source rows (no TCEQ at all), v0.4.0 has none. Expected gone.
3. **`Improved`** — v0.4.0 has *more* hours than v0.3.7 total. Refresh added data.
4. **`Match`** — v0.4.0 hours within ±`MATCH_TOLERANCE_PCT` of v0.3.7 TCEQ-only baseline.
5. **`Decreased-vs-tceq`** — v0.4.0 has fewer hours than v0.3.7 had **from TCEQ**. ⚠ This is the only verdict that means real data loss.
6. **`New-in-v040`** — only present in v0.4.0 (e.g. brand-new TCEQ feed).
7. **`Unclassified`** — fallback. Should be empty in a clean run.

In [ ]:
# Outer join on the comparison key
comp = v037.merge(
    v040[['aqsid','pollutant_group','year','v040_hours','n_v040_sources']],
    on=['aqsid','pollutant_group','year'],
    how='outer',
)
# Fill missing
for c in ['v037_epa_hours','v037_tceq_hours','v037_total_hours','v040_hours']:
    comp[c] = comp[c].fillna(0).astype(int)
comp['n_v040_sources'] = comp['n_v040_sources'].fillna(0).astype(int)

# Fill v0.4.0-only rows with site_name/county_name from the v0.4.0 side
for col in ['site_name','county_name']:
    fill = v040.set_index(['aqsid','pollutant_group','year'])[col]
    mask = comp[col].isna()
    comp.loc[mask, col] = comp.loc[mask].set_index(['aqsid','pollutant_group','year']).index.map(fill)

# Deltas vs the TCEQ baseline (the only meaningful "did we lose data" axis)
comp['delta_total']    = comp['v040_hours'] - comp['v037_total_hours']
comp['delta_vs_tceq']  = comp['v040_hours'] - comp['v037_tceq_hours']
comp['pct_delta_vs_tceq'] = np.where(
    comp['v037_tceq_hours'] > 0,
    100 * comp['delta_vs_tceq'] / comp['v037_tceq_hours'],
    np.where(comp['v040_hours'] > 0, np.inf, 0.0),
).round(2)

# ---- Verdict logic (first match wins) -----------------------------------
# 1. Out-of-scope-site      — site architecturally dropped (TSP / Von Ormy)
# 2. EPA-replaced-by-TCEQ   — v0.3.7 had EPA-only; v0.4.0 has data via TCEQ. GOOD.
# 3. EPA-only-removed       — v0.3.7 had EPA-only; v0.4.0 has nothing. Expected.
# 4. New-in-v040            — no v0.3.7 row at all; v0.4.0 has data
# 5. Match                  — v0.4.0 within ±tol of v0.3.7 TCEQ baseline
# 6. Improved               — v0.4.0 > v0.3.7 TCEQ baseline
# 7. Decreased-vs-tceq      — v0.4.0 < v0.3.7 TCEQ baseline. ⚠ Real loss
# 8. Both-empty             — both sides zero (lattice noise)
def classify(r):
    aqsid = str(r.aqsid)
    if aqsid in DROPPED_SITES:
        return 'Out-of-scope-site'
    if r.v037_total_hours > 0 and r.v037_tceq_hours == 0:
        return 'EPA-replaced-by-TCEQ' if r.v040_hours > 0 else 'EPA-only-removed'
    if r.v037_total_hours == 0 and r.v040_hours > 0:
        return 'New-in-v040'
    if r.v037_tceq_hours > 0:
        if abs(r.pct_delta_vs_tceq) <= MATCH_TOLERANCE_PCT:
            return 'Match'
        if r.delta_vs_tceq > 0:
            return 'Improved'
        return 'Decreased-vs-tceq'
    if r.v037_total_hours == 0 and r.v040_hours == 0:
        return 'Both-empty'
    return 'Unclassified'

comp['verdict'] = comp.apply(classify, axis=1)
comp = comp.sort_values(['verdict','pct_delta_vs_tceq']).reset_index(drop=True)
comp.to_csv(REPORT_DIR / 'comparison_full.csv', index=False)

print(f'Comparison rows: {len(comp):,}')
print('\nVerdict totals:')
print(comp.verdict.value_counts().to_string())
print(f'\nRow-count summary (each "row" = one (aqsid, datetime, parameter_code, poc) measurement):')
print(f'  v0.3.7 total (EPA + TCEQ):  {comp.v037_total_hours.sum():>15,}')
print(f'  v0.3.7 TCEQ portion only:   {comp.v037_tceq_hours.sum():>15,}')
print(f'  v0.3.7 EPA portion only:    {comp.v037_epa_hours.sum():>15,}')
print(f'  v0.4.0 total:               {comp.v040_hours.sum():>15,}')
print(f'  delta vs TCEQ baseline:     {comp.delta_vs_tceq.sum():>+15,}')

# Quick sanity: should be ~zero Unclassified rows in a clean run
n_unclass = int((comp.verdict == 'Unclassified').sum())
if n_unclass:
    print(f'\n⚠ {n_unclass} Unclassified rows — classifier has a gap. Inspect:')
    display(comp[comp.verdict == 'Unclassified'].head(10))

## 5. Roll-ups — old vs new by pollutant, site, and year

In [ ]:
by_pollutant = (comp.groupby('pollutant_group')
    .agg(v037_epa  = ('v037_epa_hours','sum'),
         v037_tceq = ('v037_tceq_hours','sum'),
         v037_total= ('v037_total_hours','sum'),
         v040      = ('v040_hours','sum'),
         n_groups  = ('aqsid','size'))
    .assign(delta_vs_tceq=lambda d: d.v040 - d.v037_tceq,
            pct_vs_tceq  =lambda d: np.where(d.v037_tceq>0, 100*(d.v040-d.v037_tceq)/d.v037_tceq, np.nan).round(2)))
by_pollutant.to_csv(REPORT_DIR / 'rollup_by_pollutant.csv')
print('=== Hours by pollutant_group ===')
print(by_pollutant.to_string())

In [ ]:
by_year = (comp.groupby('year')
    .agg(v037_epa  = ('v037_epa_hours','sum'),
         v037_tceq = ('v037_tceq_hours','sum'),
         v037_total= ('v037_total_hours','sum'),
         v040      = ('v040_hours','sum'))
    .assign(delta_vs_tceq=lambda d: d.v040 - d.v037_tceq,
            pct_vs_tceq  =lambda d: np.where(d.v037_tceq>0, 100*(d.v040-d.v037_tceq)/d.v037_tceq, np.nan).round(2)))
by_year.to_csv(REPORT_DIR / 'rollup_by_year.csv')
print('=== Hours by year ===')
print(by_year.to_string())

In [ ]:
by_site = (comp.groupby(['aqsid','site_name','county_name'])
    .agg(v037_epa  = ('v037_epa_hours','sum'),
         v037_tceq = ('v037_tceq_hours','sum'),
         v037_total= ('v037_total_hours','sum'),
         v040      = ('v040_hours','sum'),
         n_groups  = ('pollutant_group','nunique'),
         verdicts  = ('verdict', lambda s: '|'.join(sorted(set(s)))))
    .assign(delta_vs_tceq=lambda d: d.v040 - d.v037_tceq)
    .sort_values('delta_vs_tceq'))
by_site.to_csv(REPORT_DIR / 'rollup_by_site.csv')
print(f'=== Sites (n={len(by_site)}) — worst 15 by delta_vs_tceq (most lost) ===')
print(by_site.head(15).to_string())

## 6. Investigation queue — rows flagged Decreased-vs-tceq

**The only rows that should worry us.** Everything else is explained by an architectural decision.

In [ ]:
investigate = comp[comp.verdict == 'Decreased-vs-tceq'].copy()
investigate = investigate.sort_values('delta_vs_tceq')[
    ['aqsid','site_name','county_name','pollutant_group','year',
     'v037_epa_hours','v037_tceq_hours','v037_total_hours',
     'v040_hours','delta_vs_tceq','pct_delta_vs_tceq','n_v040_sources']
]
investigate.to_csv(REPORT_DIR / 'investigation_queue.csv', index=False)
print(f'Rows that lost TCEQ data (verdict=Decreased-vs-tceq): {len(investigate)}')
if len(investigate):
    print(f'Worst 20 (most hours lost):')
    display(investigate.head(20))
else:
    print('OK — no rows lost TCEQ data. Migration is data-clean.')

## 7. Visual diff — old vs new totals by pollutant

Side-by-side bars per pollutant group: v0.3.7 EPA portion (gray), v0.3.7 TCEQ portion (navy), v0.4.0 total (orange). When the orange bar matches or exceeds the navy bar, no TCEQ data was lost.

In [ ]:
bp = by_pollutant.copy().reset_index()
bp = bp.sort_values('v037_total', ascending=False)

fig, ax = plt.subplots(figsize=(13, 6.5))
x = np.arange(len(bp))
w = 0.27
ax.bar(x-w, bp.v037_epa,  width=w, color=BRAND_NEUTRAL, label='v0.3.7 EPA (retired)')
ax.bar(x,   bp.v037_tceq, width=w, color=BRAND_NAVY,    label='v0.3.7 TCEQ (baseline)')
ax.bar(x+w, bp.v040,      width=w, color=BRAND_ORANGE,  label='v0.4.0 total')

for i, (epa, tceq, v040) in enumerate(zip(bp.v037_epa, bp.v037_tceq, bp.v040)):
    if v040 >= tceq:
        ax.annotate('OK', (i+w, v040), ha='center', va='bottom', fontsize=8, color=BRAND_OK, fontweight='bold')
    else:
        d = v040 - tceq
        ax.annotate(f'{d:+,}', (i+w, v040), ha='center', va='bottom', fontsize=8, color=BRAND_BAD, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(bp.pollutant_group, rotation=15, ha='right')
ax.set_ylabel('hours observed (across all sites × years)')
ax.set_title('v0.3.7 vs v0.4.0 — hours by pollutant_group',
             color=BRAND_NAVY, fontweight='bold', pad=12)
ax.legend(loc='upper right', frameon=False)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'figs' / 'pollutant_diff.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Verdict breakdown: by pollutant (left) + by year (right)
VERDICT_COLORS = {
    'Match':                BRAND_OK,
    'Improved':             '#5fa9ff',
    'EPA-replaced-by-TCEQ': '#2e9bd1',
    'EPA-only-removed':     BRAND_NEUTRAL,
    'Out-of-scope-site':    '#b89b6e',
    'New-in-v040':          '#8e44ad',
    'Decreased-vs-tceq':    BRAND_BAD,
    'Both-empty':           '#e6e6e6',
    'Unclassified':         'black',
}

fig, axes = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={'width_ratios':[1.2, 2]})

v_by_p = comp.groupby(['pollutant_group','verdict']).size().unstack(fill_value=0)
v_by_p = v_by_p[[v for v in VERDICT_COLORS if v in v_by_p.columns]]
v_by_p.plot(kind='barh', stacked=True, ax=axes[0],
            color=[VERDICT_COLORS[v] for v in v_by_p.columns],
            edgecolor='white', linewidth=0.4)
axes[0].set_title('Verdict counts by pollutant_group', color=BRAND_NAVY, fontweight='bold')
axes[0].set_xlabel('# (aqsid x year) cells')
axes[0].legend(bbox_to_anchor=(1.0, 1.0), loc='upper left', fontsize=8, frameon=False)

v_by_y = comp.groupby(['year','verdict']).size().unstack(fill_value=0)
v_by_y = v_by_y[[v for v in VERDICT_COLORS if v in v_by_y.columns]]
v_by_y.plot(kind='bar', stacked=True, ax=axes[1],
            color=[VERDICT_COLORS[v] for v in v_by_y.columns],
            edgecolor='white', linewidth=0.4)
axes[1].set_title('Verdict counts by year', color=BRAND_NAVY, fontweight='bold')
axes[1].set_ylabel('# (aqsid x pollutant) cells')
axes[1].set_xlabel('')
axes[1].legend().remove()
axes[1].tick_params(axis='x', rotation=0)
for ax in axes:
    ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'figs' / 'verdict_breakdown.png', dpi=140, bbox_inches='tight')
plt.show()

## 8. Audit summary verdict

In [ ]:
from IPython.display import Markdown, display
vc = comp.verdict.value_counts().to_dict()
n_match     = vc.get('Match', 0)
n_improved  = vc.get('Improved', 0)
n_epa_repl  = vc.get('EPA-replaced-by-TCEQ', 0)
n_epa_only  = vc.get('EPA-only-removed', 0)
n_oos       = vc.get('Out-of-scope-site', 0)
n_new       = vc.get('New-in-v040', 0)
n_dec       = vc.get('Decreased-vs-tceq', 0)
n_unclass   = vc.get('Unclassified', 0)
n_total     = len(comp)
verdict_status = 'PASS' if (n_dec == 0 and n_unclass == 0) else 'INVESTIGATE'
v037_tceq_total = int(comp.v037_tceq_hours.sum())
v040_total      = int(comp.v040_hours.sum())
delta_tceq      = v040_total - v037_tceq_total
pct_delta_tceq  = 100 * delta_tceq / v037_tceq_total if v037_tceq_total else 0

md = f'''
## Audit verdict: **{verdict_status}**

### Totals (each unit = one (aqsid × datetime × parameter_code × poc) measurement row)

| Metric | Rows |
|---|---:|
| v0.3.7 EPA-sourced (retired by design) | **{int(comp.v037_epa_hours.sum()):,}** |
| v0.3.7 TCEQ-sourced (the baseline) | **{v037_tceq_total:,}** |
| v0.4.0 total (all 4 tables, folded back to v0.3.7 groups) | **{v040_total:,}** |
| Δ vs TCEQ baseline | **{delta_tceq:+,}**  ({pct_delta_tceq:+.2f}%) |

### Cell counts ({n_total:,} total (aqsid × pollutant × year) cells)

| Verdict | n | Meaning |
|---|---:|---|
| **Match** | {n_match} | v0.4.0 = v0.3.7 TCEQ within ±{MATCH_TOLERANCE_PCT}% |
| **Improved** | {n_improved} | v0.4.0 > v0.3.7 TCEQ (refresh added rows on top of existing TCEQ) |
| **EPA-replaced-by-TCEQ** | {n_epa_repl} | v0.3.7 had EPA-only data; v0.4.0 now has TCEQ rows there. **Net good** (TCEQ replaces retired EPA path) |
| **EPA-only-removed** | {n_epa_only} | v0.3.7 had EPA-only; v0.4.0 has nothing. **Expected** — site was never in TCEQ pull |
| **Out-of-scope-site** | {n_oos} | Site dropped per v0.4.0 decisions #8 / #18 (TSP-only / Von Ormy) |
| **New-in-v040** | {n_new} | Site or pollutant_group new in TCEQ pull |
| **⚠ Decreased-vs-tceq** | **{n_dec}** | TCEQ rows appear reduced — **investigate** |
| Unclassified | {n_unclass} | Classifier gap (should be 0 in a clean run) |

### Interpretation

{"**Migration is data-clean.** Every cell that shows fewer rows in v0.4.0 vs v0.3.7 is explained by an architectural decision (EPA path retired, TSP-only / Von Ormy sites dropped). The TCEQ-sourced baseline is fully preserved, and the EPA-replaced-by-TCEQ cells indicate the v0.4.0 ingest *added* TCEQ coverage where v0.3.7 had only the EPA fallback." if n_dec == 0 else f"**{n_dec} cell(s) need human review.** See `investigation_queue.csv` for the worst-loss rows. Most likely causes: (a) a TCEQ source file present in v0.3.7's ingest tree wasn't in v0.4.0's; (b) a stricter dedup rule in v0.4.0 collapsed near-duplicates that v0.3.7 retained; (c) a site's TAMIS feed changed parameter codes between runs."}

### Files in this report directory

- `comparison_full.csv` — every (aqsid × pollutant × year) cell with both sides and the verdict
- `rollup_by_pollutant.csv`, `rollup_by_year.csv`, `rollup_by_site.csv`
- `investigation_queue.csv` — Decreased-vs-tceq subset only (the action list)
- `figs/pollutant_diff.png`, `figs/verdict_breakdown.png`
'''
display(Markdown(md))

## 9. Export standalone HTML report

In [ ]:
import subprocess
candidates = [Path.cwd() / 'AM_v037_vs_v040_Audit.ipynb']
candidates += list(Path('/content').rglob('AM_v037_vs_v040_Audit.ipynb')) if Path('/content').exists() else []
notebook_path = next((p for p in candidates if p.exists()), None)
html_out = REPORT_DIR / 'AM_v037_vs_v040_Audit.html'
if notebook_path:
    cmd = [sys.executable, '-m', 'nbconvert', '--to', 'html',
           '--output', str(html_out), str(notebook_path)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print(f'OK HTML report -> {html_out}')
    else:
        print('nbconvert stderr:', res.stderr[:500])
else:
    print('Notebook path not found; skip HTML export.')

print()
print('=== All outputs ===')
for p in sorted(REPORT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(REPORT_DIR)}  ({p.stat().st_size/1024:.1f} kB)')

---

**End of v0.3.7 vs v0.4.0 audit.** This notebook is the manuscript-grade evidence that the v0.4.0 cutover preserved the TCEQ-sourced portion of the v0.3.7 dataset. Pair with `AM_Data_Availability_Audit.ipynb` (the v0.4.0-only audit) for the complete data-integrity story.